# 01 - EDA: demografia, target e missing values

## Obiettivi didattici

1. Comprendere la **struttura del dataset Diabetes 130-US Hospitals** (101.766 ricoveri, 50 feature, 1999-2008).
2. Esplorare la **distribuzione del target** `readmitted` (`<30`, `>30`, `NO`) e motivare la binarizzazione.
3. Analizzare la **demografia** (race, age, gender) e verificare se il tasso di readmission a 30 giorni differisce per sottogruppo.
4. Mappare i **valori mancanti** (codificati come `?`) e identificare le colonne da rimuovere (`weight`, `examide`, `citoglipton`).
5. Conteggiare i **pazienti con encounter multipli** (stesso `patient_nbr` con piu' ricoveri).

## Riferimenti

- Strack, B. et al. (2014). *Impact of HbA1c Measurement on Hospital Readmission Rates*. BioMed Research International.
- Fairlearn user guide: case study Diabetes 130-Hospitals.


In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

from readmit_pipeline.data import load_raw, basic_clean, missing_summary


## Caricamento del dataset grezzo

Il file `diabetic_data.csv` va scaricato manualmente dal **UCI ML Repository (id 296)** ed estratto in `data/raw/`.
`load_raw()` converte automaticamente il marker `?` in NaN.


In [ ]:
df = load_raw()
print(f'Shape: {df.shape}')
print(f'Pazienti unici: {df.patient_nbr.nunique():,}')
df.head(3)

## Distribuzione del target originale (3 classi)

Il target `readmitted` ha tre valori:
- `<30`: riammesso entro 30 giorni (classe d'interesse).
- `>30`: riammesso oltre 30 giorni.
- `NO`: non riammesso.

La classe `<30` e' fortemente sbilanciata (~11%): nessun ML "magico" ci puo' salvare da accuracy fuorviante.


In [ ]:
ax = df['readmitted'].value_counts().plot(kind='bar')
ax.set_title('Distribuzione readmission (3 classi)')
ax.set_ylabel('count')
plt.show()
print(df['readmitted'].value_counts(normalize=True).mul(100).round(2))


## Binarizzazione e cleaning

Convenzione Fairlearn: `readmitted == '<30'` -> 1, altrimenti 0.
Rimuoviamo anche colonne strutturalmente inutili (zero variance, ~97% missing) e ricoveri terminati con decesso.


In [ ]:
df_clean = basic_clean(df)
print(f'Shape post-clean: {df_clean.shape}')
rate = df_clean['readmitted_30d'].mean()
print(f'Tasso readmission <30g: {rate*100:.2f}%')


## Demografia: race, age, gender

Verifichiamo se il tasso di readmission differisce per sottogruppo. Differenze significative motivano l'audit di fairness in fase di valutazione.


In [ ]:
for col in ['race', 'age', 'gender']:
    if col not in df_clean.columns:
        continue
    rates = df_clean.groupby(col)['readmitted_30d'].agg(['mean', 'count'])
    rates['mean'] = rates['mean'].mul(100).round(2)
    rates.columns = ['readmit_rate_%', 'n']
    print(f'\n--- {col} ---')
    print(rates)


## Missing values

Le colonne con piu' missing meritano una decisione esplicita (rimuovere, imputare, raggruppare). `missing_summary` produce la tabella.


In [ ]:
missing = missing_summary(df)
missing.head(15)


## Pazienti con encounter multipli

Lo stesso `patient_nbr` puo' comparire piu' volte. Questo e' un motivo cruciale per usare un **group-aware split**: lasciare ricoveri dello stesso paziente sia in train sia in test introdurrebbe leakage.


In [ ]:
n_encounters_per_patient = df.groupby('patient_nbr').size()
print(f'Pazienti unici: {df.patient_nbr.nunique():,}')
print(f'Pazienti con >=2 ricoveri: {(n_encounters_per_patient > 1).sum():,}')
print(f'Max encounter di un singolo paziente: {n_encounters_per_patient.max()}')
